In [14]:
import pandas as pd, numpy as np
import xml.etree.ElementTree as ET

def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows).dropna(subset=["ds"])
            .sort_values("ds").reset_index(drop=True))

try:
    dis_train
except NameError:
    dis_train = load_xml_glucose("data/570-ws-training.xml")

def to_series(df):  # 5분 격자
    return (df.set_index("ds")["y"].asfreq("5min").ffill())
s_dis = to_series(dis_train)


In [15]:
def build_meal_features(index, meal_times=None):
    """
    index : DatetimeIndex(5분 고정)
    meal_times : {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}
    반환: DataFrame [since_meal_min, after_* flags, last_meal_* one-hot]
    """
    if meal_times is None:
        meal_times = {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}

    base = pd.DataFrame({"t": index})
    base = base.sort_values("t")

    feats = {}
    anchors_all = {}

    for key, hhmm in meal_times.items():
        # 각 날짜의 식사 anchor 생성
        days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                             base["t"].max().ceil("D"),
                             freq="D")
        anchors = pd.to_datetime(days.strftime("%Y-%m-%d") + " " + hhmm)
        anchors_all[key] = anchors

        # 가장 가까운 '과거' anchor 매칭
        tmp = pd.merge_asof(base, pd.DataFrame({f"{key}_anc": anchors}),
                            left_on="t", right_on=f"{key}_anc",
                            direction="backward", tolerance=pd.Timedelta("24H"))
        # 경과 분
        feats[f"since_{key}_min"] = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60

        # 구간 플래그(0~2h / 2~4h)
        feats[f"after_{key}_0_2h"] = ((feats[f"since_{key}_min"] >= 0) &
                                      (feats[f"since_{key}_min"] <= 120)).astype(int)
        feats[f"after_{key}_2_4h"] = ((feats[f"since_{key}_min"] > 120) &
                                      (feats[f"since_{key}_min"] <= 240)).astype(int)

    # '가장 최근 식사' 찾기
    last_anchor = None
    last_type = None
    for key in meal_times.keys():
        cand = base["t"] - pd.to_timedelta(feats[f"since_{key}_min"], unit="m")
        if last_anchor is None:
            last_anchor, last_type = cand, key
        else:
            mask = cand > last_anchor
            last_anchor = cand.where(~mask, cand)
            last_type   = np.where(mask, key, last_type)

    out = pd.DataFrame(feats, index=index)
    for key in meal_times.keys():
        out[f"last_is_{key}"] = (last_type == key).astype(int)

    # 대표 피처: 최근 식사로부터 경과 분(최소값)
    out["since_last_meal_min"] = np.vstack([out[f"since_{k}_min"].values
                                            for k in meal_times.keys()]).min(axis=0)
    # 음수/결측 보정
    out = out.replace([np.inf, -np.inf], np.nan).fillna(1440)
    out[out.filter(like="since_").columns] = out.filter(like="since_").clip(lower=0, upper=1440)
    return out


In [16]:
def make_supervised_with_meal(s: pd.Series, horizon_min=30, meal_times=None):
    h = horizon_min // 5
    df = pd.DataFrame({"y": s})
    # 래그(최대 120분)
    for k in [1,2,3,6,12,24]:
        df[f"lag{k}"] = df["y"].shift(k)
    # 이동평균/지수평활
    df["ma3"]   = df["y"].rolling(3,  min_periods=3).mean()
    df["ma12"]  = df["y"].rolling(12, min_periods=12).mean()
    df["ema12"] = df["y"].ewm(span=12, adjust=False).mean()
    # 캘린더
    idx = df.index
    hour = idx.hour + idx.minute/60
    df["sin_h"] = np.sin(2*np.pi*hour/24)
    df["cos_h"] = np.cos(2*np.pi*hour/24)
    df["weekday"] = idx.dayofweek
    df["is_weekend"] = df["weekday"].isin([4,5,6]).astype(int)
    # 식사 피처
    meal_feat = build_meal_features(idx, meal_times=meal_times)
    df = pd.concat([df, meal_feat], axis=1)
    # 타깃
    df["y_target"] = df["y"].shift(-h)
    df = df.dropna()
    X = df.drop(columns=["y_target","y"])
    y = df["y_target"]
    return X, y, df.index


In [17]:
def time_split_index(X, test_days=7, val_days=2):
    n_test = test_days * (24*60//5)
    n_val  = val_days  * (24*60//5)
    n = len(X)
    idx_tr = slice(0, n - (n_val + n_test))
    idx_va = slice(n - (n_val + n_test), n - n_test)
    idx_te = slice(n - n_test, n)
    return idx_tr, idx_va, idx_te


In [18]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

MEAL_TIMES = {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}  # ← 필요시 바꿔 학습

def baseline_mae(s, horizon_min, ema_span=12):
    h = horizon_min // 5
    y_true = s.shift(-h)
    mae_naive = (y_true - s).abs().dropna().mean()
    ema = s.ewm(span=ema_span, adjust=False).mean()
    mae_ema = (y_true - ema).abs().dropna().mean()
    return float(mae_naive), float(mae_ema)

def train_for_horizon(s, horizon):
    X, y, _ = make_supervised_with_meal(s, horizon_min=horizon, meal_times=MEAL_TIMES)
    idx_tr, idx_va, idx_te = time_split_index(X)
    Xtr, ytr = X.iloc[idx_tr], y.iloc[idx_tr]
    Xva, yva = X.iloc[idx_va], y.iloc[idx_va]
    Xte, yte = X.iloc[idx_te], y.iloc[idx_te]

    model = LGBMRegressor(
        n_estimators=1400, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        max_depth=-1, random_state=42, n_jobs=-1
    )
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="l1", callbacks=[])

    pred = model.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    mae_nv, mae_ema = baseline_mae(s, horizon)
    return model, {"MAE": round(mae,2), "naive": round(mae_nv,2), "EMA12": round(mae_ema,2), "n_test": len(yte)}, list(X.columns)

models, metas = {}, {}
for H in (30,60,120):
    m, score, feats = train_for_horizon(s_dis, H)
    models[H] = (m, feats)
    metas[H]  = {"horizon_min": H, "scores": score, "meal_times": MEAL_TIMES, "features": feats}
    print(f"[{H}m] LGBM MAE={score['MAE']} | naive={score['naive']} | EMA12={score['EMA12']} (n={score['n_test']})")


C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000406 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2565
[LightGBM] [Info] Number of data points in the train set: 8989, number of used features: 15
[LightGBM] [Info] Start training from score 212.067304
[30m] LGBM MAE=9.06 | naive=3.13 | EMA12=4.59 (n=2016)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001029 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2565
[LightGBM] [Info] Number of data points in the train set: 8983, number of used features: 15
[LightGBM] [Info] Start training from score 212.138595


C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))


[60m] LGBM MAE=12.89 | naive=5.23 | EMA12=6.46 (n=2016)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2566
[LightGBM] [Info] Number of data points in the train set: 8971, number of used features: 15
[LightGBM] [Info] Start training from score 212.249805


C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))


[120m] LGBM MAE=18.56 | naive=8.57 | EMA12=9.44 (n=2016)


In [19]:
# ====== 여기부터 덮어쓰기 ======
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

def train_for_horizon(s: pd.Series, horizon: int):
    # 1) 피처·타깃
    X, y, _ = make_supervised_with_meal(s, horizon_min=horizon, meal_times=MEAL_TIMES)
    idx_tr, idx_va, idx_te = time_split_index(X)
    Xtr, ytr = X.iloc[idx_tr], y.iloc[idx_tr]
    Xva, yva = X.iloc[idx_va], y.iloc[idx_va]
    Xte, yte = X.iloc[idx_te], y.iloc[idx_te]

    # 2) 제약/가중치
    mono = make_monotone_constraints(list(X.columns))   # after_*_0_2h -> +1
    sw   = make_sample_weight(Xtr)                      # 식후구간 가중치

    # 3) 모델 (✅ L2 objective + 제약 허용)
    model = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        monotone_constraints=mono,   # 제약 유지
        objective="regression"       # ✅ L2 (제약과 호환)
    )
    model.fit(
        Xtr, ytr,
        sample_weight=sw,
        eval_set=[(Xva, yva)],
        eval_metric="l1",            # ✅ 모니터링/조기중단은 MAE로
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )

    # 4) 성능
    pred = model.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    mae_nv, mae_ema = baseline_mae(s, horizon)

    score = {"MAE": round(mae, 2), "naive": round(mae_nv, 2),
             "EMA12": round(mae_ema, 2), "n_test": int(len(yte))}
    feats = list(X.columns)
    return model, score, feats

def train_and_save_all(s: pd.Series, horizons=(30, 60, 120), prefix="t1dm_lgbm"):
    results = {}
    for H in horizons:
        model, score, feats = train_for_horizon(s, H)
        meta = {"horizon_min": H, "scores": score, "meal_times": MEAL_TIMES, "features": feats}
        pkl_path = f"{prefix}_{H}m_meal.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump((model, meta), f)
        print(f"[{H}m] LGBM MAE={score['MAE']} | naive={score['naive']} | EMA12={score['EMA12']} "
              f"(n={score['n_test']}) → saved: {pkl_path}")
        results[H] = {"pkl": pkl_path, "score": score}
    return results
# ====== 여기까지 덮어쓰기 ======


In [20]:
res = train_and_save_all(s_dis, horizons=(30, 60, 120), prefix="t1dm_lgbm")
res


C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))
C:\Users\OWNER\AppData\Local\Temp\ipykernel_3560\1195534867.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  direction="backward", tolerance=pd.Timedelta("24H"))


NameError: name 'make_monotone_constraints' is not defined

In [ ]:
import pickle
for H in (30,60,120):
    with open(f"t1dm_lgbm_{H}m_meal.pkl","wb") as f:
        pickle.dump((models[H][0], metas[H]), f)
print("saved:", [f"t1dm_lgbm_{H}m_meal.pkl" for H in (30,60,120)])
